In [9]:
import pandas as pd
import re

# =====================================
# FILES
# =====================================
input_file = "deduplicated.xlsx"      # ou merged_scopus_ieee.xlsx
output_file = "screened_multimodal.xlsx"

# =====================================
# LOAD
# =====================================
if input_file.endswith(".csv"):
    df = pd.read_csv(input_file)
else:
    df = pd.read_excel(input_file, engine="openpyxl")

# =====================================
# HELPERS
# =====================================
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x

def contains_any(text, terms):
    return any(term in text for term in terms)

# =====================================
# KEYWORDS
# =====================================
vision_terms = [
    "vision", "visual", "image", "rgb", "camera", "visuo",
    "visual-tactile", "visuo-tactile"
]

tactile_terms = [
    "tactile", "haptic", "touch", "touch sensing", "tactile sensing"
]

multimodal_terms = [
    "multimodal", "multi-modal", "fusion", "cross-modal", "cross modal",
    "visuotactile", "visuo-tactile", "visual-tactile",
    "feature fusion", "cross-modal learning", "shared representation",
    "representation learning", "joint latent space", "joint embedding"
]

perception_terms = [
    "perception", "recognition", "classification", "estimation",
    "prediction", "retrieval", "representation", "understanding",
    "identification", "clustering", "attribute recognition",
    "material perception", "object recognition", "shape estimation",
    "texture recognition", "material recognition"
]

property_terms = [
    "material", "texture", "surface", "shape", "friction", "roughness",
    "hardness", "stiffness", "slip", "object properties",
    "surface properties", "physical properties", "attributes",
    "adjective", "contact force", "perceptual attributes"
]

# Exclusions fortes
hard_exclude_terms = {
    "review article": ["review", "survey"],
    "human study / neuroscience": [
        "electroencephalography", "eeg", "erp", "sep", "participant",
        "participants", "consumer", "self-reported", "gustatory",
        "dining environments", "psychophysical"
    ],
    "vr/ar or immersive communication": [
        "virtual reality", "augmented reality", "immersive communication"
    ],
    "pure hardware / fabrication": [
        "fabrication", "floating gate transistor", "energy storage",
        "neuromorphic hardware", "artificial sensory neurons",
        "mxene", "microbattery", "supercapacitor"
    ],
    "communications / compression": [
        "compression", "coding", "bitrate", "content delivery",
        "wireless communications"
    ],
    "medical or unrelated application": [
        "breast tumors", "visually impaired", "daily activity tracking",
        "wearable sensors"
    ]
}

# Biais conception/design : pénalité légère, pas exclusion automatique
design_bias_terms = [
    "sensor design", "tactile sensor", "sensor named", "proposes a sensor",
    "large-area tactile sensing", "total internal reflection mechanism",
    "hardware level", "drive strategy", "spatial resolution",
    "sensor fabrication", "device design"
]

# =====================================
# EVALUATION
# =====================================
def evaluate_row(row):
    title = clean_text(row.get("title", ""))
    abstract = clean_text(row.get("abstract", ""))
    keywords = clean_text(row.get("keywords", ""))
    doc_type = clean_text(row.get("documentType", ""))

    text = " ".join([title, abstract, keywords, doc_type])

    score = 0
    reasons = []

    has_vision = contains_any(text, vision_terms)
    has_tactile = contains_any(text, tactile_terms)
    has_multimodal = contains_any(text, multimodal_terms)
    has_perception = contains_any(text, perception_terms)
    has_property = contains_any(text, property_terms)
    has_design_bias = contains_any(text, design_bias_terms)

    # =====================================
    # EXCLUSIONS FORTES
    # =====================================
    for reason, terms in hard_exclude_terms.items():
        if contains_any(text, terms):
            if reason == "review article":
                if "review" in doc_type or title.startswith("review") or " a review" in title:
                    return pd.Series(["exclude", reason, 0])
            else:
                return pd.Series(["exclude", reason, 0])

    # =====================================
    # SCORE POSITIF
    # =====================================
    if has_vision:
        score += 2
        reasons.append("vision")

    if has_tactile:
        score += 2
        reasons.append("tactile")

    if has_multimodal:
        score += 2
        reasons.append("multimodal/cross-modal")

    if has_perception:
        score += 2
        reasons.append("perception task")

    if has_property:
        score += 2
        reasons.append("object/material properties")

    # pénalité légère pour orientation design
    if has_design_bias:
        score -= 2
        reasons.append("design-oriented")

    # borne 0..10
    score = max(0, min(10, score))

    # =====================================
    # DECISION
    # =====================================
    if score >= 7 and has_vision and has_tactile:
        if has_perception:
            decision = "include"
        else:
            decision = "maybe"
            reasons.append("perception not explicit")
    elif score >= 4:
        decision = "maybe"
    else:
        decision = "exclude"

    # Si très design-oriented, on évite include direct
    if decision == "include" and has_design_bias:
        decision = "maybe"
        reasons.append("manual check: perception vs design")

    reason_text = ", ".join(reasons) if reasons else "low relevance"

    return pd.Series([decision, reason_text, score])

# =====================================
# APPLY
# =====================================
df[["Decision", "Reason for decision", "Score /10"]] = df.apply(evaluate_row, axis=1)

# tri utile
decision_order = {"include": 0, "maybe": 1, "exclude": 2}
df["__sort"] = df["Decision"].map(decision_order).fillna(9)
df = df.sort_values(["__sort", "Score /10"], ascending=[True, False]).drop(columns="__sort")

# =====================================
# SAVE
# =====================================
if output_file.endswith(".csv"):
    df.to_csv(output_file, index=False)
else:
    df.to_excel(output_file, index=False)

# =====================================
# STATS
# =====================================
print("Done:", output_file)
print("\nDecision counts:")
print(df["Decision"].value_counts())

print("\nAverage score by decision:")
print(df.groupby("Decision")["Score /10"].mean().round(2))

print("\nTop include papers:")
display(df[df["Decision"] == "include"][["title", "Decision", "Score /10", "Reason for decision"]].head(10))

Done: screened_multimodal.xlsx

Decision counts:
exclude    72
maybe      39
include    23
Name: Decision, dtype: int64

Average score by decision:
Decision
exclude    0.22
include    9.48
maybe      6.15
Name: Score /10, dtype: float64

Top include papers:


,title,Decision,Score /10,Reason for decision
0,Vision2Touch: Imaging Estimation of Surface Ta...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
6,Surface Material Recognition Using Active Mult...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
10,Visual tactile fusion object clustering,include,10,"vision, tactile, multimodal/cross-modal, perce..."
11,Multimodal fusion recognition for digital twin,include,10,"vision, tactile, multimodal/cross-modal, perce..."
25,Cascade broad learning for multi-modal materia...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
28,Classification of Visual-Tactile Fusion Object...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
48,VikitaFusion: Object Recognition Based on Hete...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
50,TL-SNN: event-driven visual-tactile learning w...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
65,Bilinear Feature Fusion Convolutional Neural N...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
69,Visual-tactile fusion learning for material re...,include,10,"vision, tactile, multimodal/cross-modal, perce..."
